### 📈 How to Trade Options with the Black-Scholes Model

##### ▶️ Related Quant Guild Videos:

- [Quant Explains Risk-Neutral Option Pricing](https://youtu.be/wYpg0TGxvgM)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

- [How to Trade the Covered Call](https://youtu.be/iPsPRQlDeTA)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

In [ ]:
%%html
<style>
/* Overwrite the hard-coded white background for ipywidgets */
.cell-output-ipywidget-background {
    background-color: transparent !important;
}
/* Set widget foreground text and color to match the VS Code dark theme */
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>

### 📖 Sections

#### 1.) 📉 Black-Scholes Model

- Relative Option Pricing and Implied Volatility

- Delta and Probability of "Making Money"

#### 2.) 💭 Closing Thoughts and Future Topics

---

##### What sense can we make of price if it's apples to oranges?

The price of an ATM 15 day GOOG call is \$11.00

The price of an ATM 15 day NVDA call is \$11.00

*Which is more expensive?*

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 1. Parameters & Data Generation ---
T = 1.0         # Time to maturity (1 year)
r = 0.05        # Risk-free rate

# Spot Prices (Actuals for the right chart)
S0_nvda = 214
S0_goog = 340

# Normalized Base for Left Chart (Spot and Strike rebased to 100)
S0_norm = 100
K_norm = 100    

# Volatility setup (GOOG has higher variance)
vol_nvda = 0.25
vol_goog = 0.50

n_steps = 100   # Number of simulation steps
dt = T / n_steps

def call_payoff(S, K):
    return np.maximum(S - K, 0)

# Generate Price Paths
np.random.seed(42) # Fixed seed for reproducible display
dates = pd.date_range(start='2025-01-01', periods=n_steps+1, freq='B')

# NVDA Path (Actual & Normalized)
rets_nvda = np.random.normal((r - 0.5 * vol_nvda**2) * dt, vol_nvda * np.sqrt(dt), n_steps)
path_nvda = S0_nvda * np.exp(np.cumsum(rets_nvda))
path_nvda = np.insert(path_nvda, 0, S0_nvda)
path_nvda_norm = (path_nvda / S0_nvda) * S0_norm

# GOOG Path (Actual & Normalized)
rets_goog = np.random.normal((r - 0.5 * vol_goog**2) * dt, vol_goog * np.sqrt(dt), n_steps)
path_goog = S0_goog * np.exp(np.cumsum(rets_goog))
path_goog = np.insert(path_goog, 0, S0_goog)
path_goog_norm = (path_goog / S0_goog) * S0_norm

# Dynamic boundaries for plotting (based on normalized data for the left side)
x_min_payoff = min(np.min(path_nvda_norm), np.min(path_goog_norm), K_norm) * 0.85
x_max_payoff = max(np.max(path_nvda_norm), np.max(path_goog_norm), K_norm * 1.1) * 1.05
y_max_payoff = call_payoff(x_max_payoff, K_norm) * 1.1

S_range = np.linspace(x_min_payoff, x_max_payoff, 200)
payoff_vals = call_payoff(S_range, K_norm)

# Colors
off_white = "#e0e0e0"
strike_color = "#ff4b4b"
nvda_color = "#00ff88"  # Green
goog_color = "#00d1ff"  # Blue

# --- 2. Figure Setup ---
# Enable secondary_y for the second column
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Normalized Call Payoff Diagram", "Underlying Price Simulation"),
    horizontal_spacing=0.15,
    specs=[[{"secondary_y": False}, {"secondary_y": True}]]
)

# Base traces (Order strictly matters for animation frames)

# 1. Payoff curve (Left)
fig.add_trace(go.Scatter(x=S_range, y=payoff_vals, line=dict(color=off_white, width=3), showlegend=False), row=1, col=1)
# 2. Strike line (Left - normalized)
fig.add_trace(go.Scatter(x=[K_norm, K_norm], y=[0, y_max_payoff], line=dict(color=strike_color, width=1, dash='dash'), showlegend=False), row=1, col=1)
# 3. NVDA normalized price line (Left)
fig.add_trace(go.Scatter(x=[S0_norm, S0_norm], y=[0, y_max_payoff], line=dict(color=nvda_color, width=2), mode='lines', showlegend=False), row=1, col=1)
# 4. GOOG normalized price line (Left)
fig.add_trace(go.Scatter(x=[S0_norm, S0_norm], y=[0, y_max_payoff], line=dict(color=goog_color, width=2), mode='lines', showlegend=False), row=1, col=1)

# 5. NVDA Strike line (Right, attached to primary y-axis)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda], line=dict(color=nvda_color, width=1, dash='dash'), opacity=0.5, showlegend=False), row=1, col=2, secondary_y=False)
# 6. GOOG Strike line (Right, attached to secondary y-axis)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_goog, S0_goog], line=dict(color=goog_color, width=1, dash='dash'), opacity=0.5, showlegend=False), row=1, col=2, secondary_y=True)
# 7. NVDA path (Right, primary y-axis) - ONLY THIS IN LEGEND
fig.add_trace(go.Scatter(x=[dates[0]], y=[S0_nvda], line=dict(color=nvda_color, width=2.5), mode='lines', name="NVDA Spot", showlegend=True), row=1, col=2, secondary_y=False)
# 8. GOOG path (Right, secondary y-axis) - ONLY THIS IN LEGEND
fig.add_trace(go.Scatter(x=[dates[0]], y=[S0_goog], line=dict(color=goog_color, width=2.5), mode='lines', name="GOOG Spot", showlegend=True), row=1, col=2, secondary_y=True)

# --- 3. Animation Frames & Slider Steps ---
frames = []
slider_steps = []

for i in range(n_steps + 1):
    # Normalized prices for the left chart
    curr_nvda_norm = path_nvda_norm[i]
    curr_goog_norm = path_goog_norm[i]
    
    frame_name = f"f{i}"
    
    # Create Frame updating all 8 traces
    frames.append(go.Frame(
        data=[
            go.Scatter(x=S_range, y=payoff_vals),                           # 1
            go.Scatter(x=[K_norm, K_norm], y=[0, y_max_payoff]),            # 2
            go.Scatter(x=[curr_nvda_norm, curr_nvda_norm], y=[0, y_max_payoff]), # 3
            go.Scatter(x=[curr_goog_norm, curr_goog_norm], y=[0, y_max_payoff]), # 4
            go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda]),      # 5
            go.Scatter(x=[dates[0], dates[-1]], y=[S0_goog, S0_goog]),      # 6
            go.Scatter(x=dates[:i+1], y=path_nvda[:i+1]),                   # 7
            go.Scatter(x=dates[:i+1], y=path_goog[:i+1])                    # 8
        ],
        name=frame_name
    ))
    
    # Create Slider Step
    step = {
        "args": [
            [frame_name],
            {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}
        ],
        "label": str(i),
        "method": "animate"
    }
    slider_steps.append(step)

fig.frames = frames

# --- 4. Layout with Buttons and Slider ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=650, width=1200,
    margin=dict(t=100, b=150, r=50), 
    # Overlay the legend in the upper right corner of the right chart area
    legend=dict(
        x=0.97,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor='rgba(30,30,30,0.8)',
        bordercolor='rgba(255,255,255,0.25)',
        borderwidth=1
    ), 
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 20, "redraw": False}, "fromcurrent": True}]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Step: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

axis_style = dict(
    showgrid=True, gridcolor='rgba(255,255,255,0.1)',
    tickfont=dict(color=off_white), linecolor=off_white,
    zeroline=False, title_font=dict(color=off_white)
)

# Left Plot Axes (Normalized)
fig.update_xaxes(axis_style, range=[x_min_payoff, x_max_payoff], title_text="Relative Asset Price (Base 100)", row=1, col=1)
fig.update_yaxes(axis_style, range=[-5, y_max_payoff], title_text="Relative Payoff", row=1, col=1)

# Right Plot X-Axis
fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=2)

# Right Plot Twin Y-Axes (Color-coded actuals)
fig.update_yaxes(axis_style, range=[np.min(path_nvda)*0.9, np.max(path_nvda)*1.1], title_text="NVDA Price ($)", title_font=dict(color=nvda_color), tickfont=dict(color=nvda_color), row=1, col=2, secondary_y=False)
fig.update_yaxes(axis_style, range=[np.min(path_goog)*0.9, np.max(path_goog)*1.1], title_text="GOOG Price ($)", title_font=dict(color=goog_color), tickfont=dict(color=goog_color), showgrid=False, row=1, col=2, secondary_y=True)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Supply and demand create an equilibrium price, and the Black-Scholes model looks to explain the way markets are currently pricing these contracts.


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
from scipy.optimize import brentq

# --- 1. Parameters & IV Calculation ---
T_days = 15
T = T_days / 365.0  # Time to maturity in years
r = 0.04            # Risk-free rate (4%)

S0_nvda = 214.0
S0_goog = 340.0
market_price = 11.00 # ATM Call price for both

# Black-Scholes Call Price Formula
def bs_call(S, K, T, r, vol):
    d1 = (np.log(S / K) + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Root finder to back out Implied Volatility
def implied_volatility(price, S, K, T, r):
    objective = lambda vol: bs_call(S, K, T, r, vol) - price
    return brentq(objective, 1e-4, 5.0) # Search between 0.01% and 500% IV

iv_nvda = implied_volatility(market_price, S0_nvda, S0_nvda, T, r)
iv_goog = implied_volatility(market_price, S0_goog, S0_goog, T, r)

print(f"Calculated NVDA IV: {iv_nvda:.2%}")
print(f"Calculated GOOG IV: {iv_goog:.2%}")

# --- 2. Data Generation (Monte Carlo Matrix) ---
n_paths = 50
n_steps = T_days
dt = T / n_steps

np.random.seed(42) # Fixed seed for reproducible display
dates = pd.date_range(start=pd.Timestamp.today(), periods=n_steps+1, freq='B')

def generate_paths(S0, vol, n_paths, n_steps, dt, r):
    paths = np.zeros((n_steps + 1, n_paths))
    paths[0] = S0
    for i in range(1, n_steps + 1):
        Z = np.random.standard_normal(n_paths)
        paths[i] = paths[i-1] * np.exp((r - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * Z)
    return paths

# Generate actual price paths
paths_nvda = generate_paths(S0_nvda, iv_nvda, n_paths, n_steps, dt, r)
paths_goog = generate_paths(S0_goog, iv_goog, n_paths, n_steps, dt, r)

# Colors and randomized opacities
off_white = "#e0e0e0"
nvda_color = "#00ff88"  # Green
goog_color = "#00d1ff"  # Blue
opacities = np.random.uniform(0.15, 0.70, n_paths)

# Static Y-axis limits so the animation doesn't jitter
y_min_nvda, y_max_nvda = np.min(paths_nvda) * 0.95, np.max(paths_nvda) * 1.05
y_min_goog, y_max_goog = np.min(paths_goog) * 0.95, np.max(paths_goog) * 1.05

# --- 3. Figure Setup ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"NVDA Simulated Paths (IV: {iv_nvda:.1%})", 
        f"GOOG Simulated Paths (IV: {iv_goog:.1%})"
    ),
    horizontal_spacing=0.10
)

# --- Base Traces (Strict Order Required for Frames) ---
# Trace 0: NVDA horizontal start line (Left)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda], line=dict(color=off_white, width=1, dash='dash'), opacity=0.5, name="NVDA Spot"), row=1, col=1)
# Trace 1: GOOG horizontal start line (Right)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_goog, S0_goog], line=dict(color=off_white, width=1, dash='dash'), opacity=0.5, name="GOOG Spot"), row=1, col=2)

# Traces 2 to 51: NVDA Paths (Left)
for i in range(n_paths):
    fig.add_trace(go.Scatter(x=[dates[0]], y=[S0_nvda], line=dict(color=nvda_color, width=1.5), opacity=opacities[i], showlegend=False), row=1, col=1)

# Traces 52 to 101: GOOG Paths (Right)
for i in range(n_paths):
    fig.add_trace(go.Scatter(x=[dates[0]], y=[S0_goog], line=dict(color=goog_color, width=1.5), opacity=opacities[i], showlegend=False), row=1, col=2)


# --- 4. Animation Frames & Slider Steps ---
frames = []
slider_steps = []

for k in range(n_steps + 1):
    frame_data = []
    
    # 0-1: Static horizontal lines
    frame_data.append(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda]))
    frame_data.append(go.Scatter(x=[dates[0], dates[-1]], y=[S0_goog, S0_goog]))
    
    # 2-51: NVDA Path updates
    for i in range(n_paths):
        frame_data.append(go.Scatter(x=dates[:k+1], y=paths_nvda[:k+1, i]))
        
    # 52-101: GOOG Path updates
    for i in range(n_paths):
        frame_data.append(go.Scatter(x=dates[:k+1], y=paths_goog[:k+1, i]))

    frame_name = f"f{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    step = {
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}],
        "label": str(k),
        "method": "animate"
    }
    slider_steps.append(step)

fig.frames = frames

# --- 5. Layout with Buttons and Slider ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=600, width=1200,
    margin=dict(t=100, b=150, l=50, r=50), 
    legend=dict(x=0.5, y=-0.2, xanchor='center', orientation='h', bgcolor='rgba(0,0,0,0)'),
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                # Slightly slower duration (100ms) to cleanly render 100 simultaneous paths
                "args": [None, {"frame": {"duration": 100, "redraw": False}, "fromcurrent": True}] 
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Day: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

axis_style = dict(
    showgrid=True, gridcolor='rgba(255,255,255,0.1)',
    tickfont=dict(color=off_white), linecolor=off_white,
    zeroline=False, title_font=dict(color=off_white)
)

# Set static ranges and styling
fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=1)
fig.update_yaxes(axis_style, range=[y_min_nvda, y_max_nvda], title_text="NVDA Price ($)", row=1, col=1)

fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=2)
fig.update_yaxes(axis_style, range=[y_min_goog, y_max_goog], title_text="GOOG Price ($)", row=1, col=2)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### In the Black-Scholes Framework Delta is Roughly Probability of Expiring ITM

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import norm
from scipy.optimize import brentq

# --- 1. Parameters, IV & Delta Calculation ---
T_days = 15
T = T_days / 365.0  # Time to maturity in years
r = 0.04            # Risk-free rate (4%)
S0_nvda = 214.0
market_price = 11.00 # ATM Call price

# Black-Scholes Call Price Formula
def bs_call(S, K, T, r, vol):
    d1 = (np.log(S / K) + (r + 0.5 * vol**2) * T) / (vol * np.sqrt(T))
    d2 = d1 - vol * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

# Root finder to back out Implied Volatility
def implied_volatility(price, S, K, T, r):
    objective = lambda vol: bs_call(S, K, T, r, vol) - price
    return brentq(objective, 1e-4, 5.0)

# 1a. Back out IV
iv_nvda = implied_volatility(market_price, S0_nvda, S0_nvda, T, r)

# 1b. Calculate ATM Delta
d1_nvda = (np.log(S0_nvda / S0_nvda) + (r + 0.5 * iv_nvda**2) * T) / (iv_nvda * np.sqrt(T))
delta_nvda = norm.cdf(d1_nvda)

print(f"Calculated NVDA IV: {iv_nvda:.2%}")
print(f"Calculated NVDA Delta: {delta_nvda:.4f}")

# --- 2. Data Generation (Monte Carlo Matrix) ---
n_paths = 250
n_steps = T_days
dt = T / n_steps

np.random.seed(42) # Fixed seed for reproducible display
dates = pd.date_range(start=pd.Timestamp.today(), periods=n_steps+1, freq='B')

# Generate paths
paths_nvda = np.zeros((n_steps + 1, n_paths))
paths_nvda[0] = S0_nvda
for i in range(1, n_steps + 1):
    Z = np.random.standard_normal(n_paths)
    paths_nvda[i] = paths_nvda[i-1] * np.exp((r - 0.5 * iv_nvda**2) * dt + iv_nvda * np.sqrt(dt) * Z)

# Pre-calculate ITM/OTM outcomes for the bar chart
final_prices = paths_nvda[-1, :]
is_itm = final_prices > S0_nvda

# Colors and randomized opacities
off_white = "#e0e0e0"
nvda_color = "#00ff88"  # Green for ITM
otm_color = "#ff4b4b"   # Red for OTM
opacities = np.random.uniform(0.15, 0.70, n_paths)

# Static limits to prevent axis jitter
y_min_nvda, y_max_nvda = np.min(paths_nvda) * 0.95, np.max(paths_nvda) * 1.05

# --- 3. Figure Setup ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"NVDA Simulated Paths (Delta: {delta_nvda:.2f})", 
        "Path Outcomes (Convergence to Delta)"
    ),
    horizontal_spacing=0.10
)

# --- Base Traces (Strict Order Required for Frames) ---
# Trace 0: Strike line (Left)
fig.add_trace(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda], line=dict(color=off_white, width=1, dash='dash'), opacity=0.5, name="Strike"), row=1, col=1)

# Traces 1 to 50: Empty placeholders for the paths to be drawn into
for i in range(n_paths):
    fig.add_trace(go.Scatter(x=[None], y=[None], line=dict(color=nvda_color, width=1.5), opacity=opacities[i], showlegend=False), row=1, col=1)

# Trace 51: Empty bar chart
fig.add_trace(go.Bar(
    x=["ITM (> Strike)", "OTM (<= Strike)"], 
    y=[0, 0], 
    marker_color=[nvda_color, otm_color], 
    text=["0%", "0%"], 
    textposition="auto",
    textfont=dict(color="black", size=14, weight="bold"),
    showlegend=False
), row=1, col=2)


# --- 4. Animation Frames & Slider Steps ---
frames = []
slider_steps = []

# Iterate Path-by-Path (not step-by-step)
for k in range(n_paths):
    frame_data = []
    
    # 0: Keep static strike line
    frame_data.append(go.Scatter(x=[dates[0], dates[-1]], y=[S0_nvda, S0_nvda]))
    
    # 1-50: Render paths up to 'k', leave the rest empty
    for i in range(n_paths):
        if i <= k:
            frame_data.append(go.Scatter(x=dates, y=paths_nvda[:, i]))
        else:
            frame_data.append(go.Scatter(x=[None], y=[None]))
            
    # 51: Update Bar Chart tally based on the subset of paths drawn so far
    itm_count = np.sum(is_itm[:k+1])
    otm_count = (k + 1) - itm_count
    itm_pct = itm_count / (k + 1)
    otm_pct = otm_count / (k + 1)
    
    frame_data.append(go.Bar(
        x=["ITM (> Strike)", "OTM (<= Strike)"], 
        y=[itm_count, otm_count], 
        text=[f"{itm_pct:.1%}", f"{otm_pct:.1%}"]
    ))

    frame_name = f"f{k}"
    frames.append(go.Frame(data=frame_data, name=frame_name))
    
    step = {
        "args": [[frame_name], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}],
        "label": str(k + 1), # 1-indexed for the slider
        "method": "animate"
    }
    slider_steps.append(step)

fig.frames = frames

# --- 5. Layout with Buttons and Slider ---
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=600, width=1200,
    margin=dict(t=100, b=150, l=50, r=50), 
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [None, {"frame": {"duration": 200, "redraw": False}, "fromcurrent": True}] 
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "fromcurrent": True}]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Simulated Paths: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

axis_style = dict(
    showgrid=True, gridcolor='rgba(255,255,255,0.1)',
    tickfont=dict(color=off_white), linecolor=off_white,
    zeroline=False, title_font=dict(color=off_white)
)

# Left Plot: NVDA Price Paths
fig.update_xaxes(axis_style, range=[dates[0], dates[-1]], title_text="Date", row=1, col=1)
fig.update_yaxes(axis_style, range=[y_min_nvda, y_max_nvda], title_text="NVDA Price ($)", row=1, col=1)

# Right Plot: Bar Chart
fig.update_xaxes(axis_style, title_text="Outcome Status at Maturity", row=1, col=2)
# Set fixed Y-axis limit to max paths so the bars grow dynamically upward instead of axis jumping
fig.update_yaxes(axis_style, range=[0, n_paths], title_text="Number of Paths", row=1, col=2)

fig.show()

---

#### 2.) 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary**

  - We explored the Black-Scholes model to build intuition about its applications and limitations in the real world, emphasizing that it is not intended as a forecasting or prediction model for asset prices or option outcomes.
  - A key takeaway is that Black-Scholes option prices between different contexts are not always apples-to-apples; differences in inputs, market assumptions, and volatility surfaces mean prices can't be universally compared, unlike implied volatility (iVol), which offers a consistent, relative scale for comparing options across strikes and maturities.
  - We clarified that while the Black-Scholes “delta” is often associated with probability, in reality it is a hedge ratio—it does not represent the actual chance of finishing in-the-money, but instead indicates how much the option price changes with small moves in the underlying.
  - Through practical examples and visualizations, we highlighted why it's crucial for traders to interpret model outputs with caution and always consider the assumptions, including market dynamics and model risk, when applying Black-Scholes-based insights to real-world option trading.

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$